# MCM LTER ALMS Data Processing

This script was created by Jared Collins in 2025 to be used by the MCM LTER stream team as an easy way to organize data from the Active Layer Monitoring Stations (ALMS). The stations measuring volumetric soil content (VWC), conductivity, and temperature at 5 different locations across the stream channel.

## Importing Packages + Setting Directory

It is very important that the computer being used for processing contains the proper packages, paths, and names for the data files that will be used. The hyporheic data is a little bit more difficult to work with than data processed in Aquarius- unlike data from Aquarius, it is not automatically converted into a dataframe from the .dat file.

**To use this script, please copy the .dat file you would like to convert to your downloads folder.**

Note: The datafiles from the CR1000X all have the same name with no date differentiation, so this can only be done using one data file per stream at a time.

In [1]:
from datetime import date
import pandas as pd
import os
from pathlib import Path
import re
import numpy as np

In [2]:
home = Path.home()
downloads_path = home / "Downloads"
downloads_path = downloads_path.as_posix()
print("Downloads folder:", downloads_path)

Downloads folder: C:/Users/jared/Downloads


In [106]:
almsid_map = {
    'F6': 'f6_alms_ALMS_F6',
    'GC': 'F9_ALMS_ALMS_GC_data',
    'VG': 'VG_ALMS_ALMS_VG_data',
    'WHC': 'whc alms_ALMS_WHC_data',
    'WTB': 'wtb_alms_ALMS_WTB_data'}

station_map = {
    'F6': '01',
    'GC': '04',
    'VG': '03',
    'WHC': '02',
    'WTB': '06'}

location_map = {
    'F6': '',
    'GC': 'Green Creek',
    'VG': 'Von Guerard',
    'WHC': 'Wormherder Creek',
    'WTB': 'Watertrack B'}

vg_map = {
    '1': 'a_35',
    '2': 'a_15',
    '3': 'a_5',
    '4': 'b_33',
    '5': 'b_15',
    '6': 'b_5',
    '7': 'c_35',
    '8': 'c_15',
    '9': 'c_5',
    '10': 'd_25',
    '11': 'd_15',
    '12': 'd_5',
    '13': 'e_25',
    '14': 'e_15',
    '15': 'e_5'}

vgtc_map = {
    '1': 'a_40',
    '2': 'a_30',
    '3': 'a_25',
    '4': 'a_10',
    '5': 'a_1',
    '6': 'b_40',
    '7': 'b_30',
    '8': 'b_25',
    '9': 'b_10',
    '10': 'b_1',
    '11': 'c_40',
    '12': 'c_30',
    '13': 'c_25',
    '14': 'c_10',
    '15': 'c_1',
    '16': 'd_60',
    '17': 'd_50',
    '18': 'd_40',
    '19': 'd_30',
    '20': 'd_10',
    '21': 'e_30',
    '22': 'e_20',
    '23': 'e_10'}  

f9_map = {
    '1': 'a_35',
    '2': 'a_15',
    '3': 'a_5',
    '4': 'b_35',
    '5': 'b_15',
    '6': 'b_5',
    '7': 'c_35',
    '8': 'c_15',
    '9': 'c_5',
    '10': 'd_35',
    '11': 'd_15',
    '12': 'd_5',
    '13': 'e_30',
    '14': 'e_15',
    '15': 'e_5'}

f9tc_map = {
    '1': 'a_30',
    '2': 'a_25',
    '3': 'a_20',
    '4': 'a_10',
    '5': 'a_1',
    '6': 'b_30',
    '7': 'b_25',
    '8': 'b_20',
    '9': 'b_10',
    '10': 'b_1',
    '11': 'c_45',
    '12': 'c_30',
    '13': 'c_25',
    '14': 'c_20',
    '15': 'c_10',
    '16': 'c_1 ',
    '17': 'd_45',
    '18': 'd_30',
    '19': 'd_25',
    '20': 'd_20',
    '21': 'd_10',
    '22': 'd_1',
    '23': 'e_20',
    '24': 'e_10',
    '25': 'e_1'}  

In order for this script to work properly, you will need to input necessary information about the files that you are hoping to publish:
- ALMS ID: F6, GC, VG, WHC, WTB
- Season: The two years of the season (for example, 2024-2025 season is written as 2425)

**Edit the following cell with the information above. If done correctly, this is the only cell you will have to edit.**

In [107]:
almsid = 'GC'
season = '2425'

In [108]:
alms = pd.read_csv(f'{downloads_path}/{almsid_map[almsid]}.dat', skiprows = [0,2,3], sep = ',', low_memory = False)

def build_dataset(df, almsid, station_map, location_map, measurement):
    result = pd.DataFrame()
    result['date_time'] = df['TIMESTAMP']
    result['dataset_code'] = f'ALMS{station_map[almsid]}_{almsid}'
    result['station'] = f'ALMS{station_map[almsid]}'
    result['location'] = location_map[almsid]
    result['measurement'] = measurement
    result = result[['dataset_code', 'date_time', 'station', 'location', 'measurement']]
    return result

def build_vwc(df, result, almsid):
    map_name = f"{almsid.lower()}_map"
    if map_name not in globals():
        raise ValueError(f"Mapping '{map_name}' not found. Make sure it is defined.")
    mapping = globals()[map_name]
    for key, new_name in mapping.items():
        raw_col = f"VWCm({key})"
        result[new_name] = df[raw_col]
    mapped_cols = list(mapping.values())
    def sort_key(col):
        letter = col[0]
        depth = int(col.split("_")[1])
        return (letter, depth)
    sorted_cols = sorted(mapped_cols, key=sort_key)
    base_cols = [c for c in result.columns if c not in sorted_cols]
    result = result[base_cols + sorted_cols]
    return result

def build_ec(df, result, almsid):
    map_name = f"{almsid.lower()}_map"
    if map_name not in globals():
        raise ValueError(f"Mapping '{map_name}' not found. Make sure it is defined.")
    mapping = globals()[map_name]
    for key, new_name in mapping.items():
        raw_col = f"ECp({key})"
        result[new_name] = df[raw_col]
    mapped_cols = list(mapping.values())
    def sort_key(col):
        letter = col[0]
        depth = int(col.split("_")[1])
        return (letter, depth)
    sorted_cols = sorted(mapped_cols, key=sort_key)
    base_cols = [c for c in result.columns if c not in sorted_cols]
    result = result[base_cols + sorted_cols]
    return result

def build_temp(df, result, almsid):
    map_name = f"{almsid.lower()}_map"
    tc_map_name = f"{almsid.lower()}tc_map"
    if map_name not in globals():
        raise ValueError(f"Mapping '{map_name}' not found.")
    if tc_map_name not in globals():
        raise ValueError(f"Mapping '{tc_map_name}' not found.")
    temp_map = globals()[map_name] 
    tc_map = globals()[tc_map_name]
    for raw_key, new_name in temp_map.items():
        raw_col = f"Temp({raw_key})"
        result[new_name] = df[raw_col]
    for raw_key, new_name in tc_map.items():
        raw_col = f"ALMS_{almsid}_TC({raw_key})"
        result[new_name] = df[raw_col]
    all_mapped_cols = list(temp_map.values()) + list(tc_map.values())
    def sort_key(col):
        letter = col[0]                # 'a', 'b', 'c', ...
        depth = int(col.split("_")[1]) # numeric depth
        return (letter, depth)
    sorted_cols = sorted(all_mapped_cols, key=sort_key)
    base_cols = [c for c in result.columns if c not in sorted_cols]
    return result[base_cols + sorted_cols]

def clean_sensor_failures(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    bad_values = [7999, -7999, 9999, -9999, '7999', '-7999', '9999', '-9999', '-INF', 'INF','NAN','nan']
    df.replace(bad_values, np.nan, inplace=True)
    df = df.replace([np.inf, -np.inf], np.nan)
    for col in df.select_dtypes(include=[np.number]).columns:
        df[col] = df[col].astype(float)
    return df

In [109]:
vwc_subm = build_dataset(df = alms, almsid = almsid, station_map = station_map, location_map = location_map, measurement = 'vwc')
vwc_subm = build_vwc(df = alms, result = vwc_subm, almsid = almsid)
vwc_subm = clean_sensor_failures(vwc_subm)

ec_subm = build_dataset(df = alms, almsid = almsid, station_map = station_map, location_map = location_map, measurement = 'ec')
ec_subm = build_ec(df = alms, result = vwc_subm, almsid = almsid)
ec_subm = clean_sensor_failures(ec_subm)

temp_subm = build_dataset(df = alms, almsid = almsid, station_map = station_map, location_map = location_map, measurement = 'temp')
temp_subm = build_temp(df = alms, result = temp_subm, almsid = almsid)
temp_subm = clean_sensor_failures(temp_subm)

vwc_subm.to_csv(f'{downloads_path}/ALMS{station_map[almsid]}_{almsid}_VWC_{season}_SUBM.csv', index = False, na_rep = '')
ec_subm.to_csv(f'{downloads_path}/ALMS{station_map[almsid]}_{almsid}_EC_{season}_SUBM.csv', index = False, na_rep = '')
temp_subm.to_csv(f'{downloads_path}/ALMS{station_map[almsid]}_{almsid}_TEMP_{season}_SUBM.csv', index = False, na_rep = '')